In [1]:
!pip uninstall -y torch torchvision torchaudio transformers openai
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 transformers openai==0.28


Found existing installation: torch 2.5.1
Uninstalling torch-2.5.1:
  Successfully uninstalled torch-2.5.1
Found existing installation: torchvision 0.20.1
Uninstalling torchvision-0.20.1:
  Successfully uninstalled torchvision-0.20.1
Found existing installation: torchaudio 2.5.1
Uninstalling torchaudio-2.5.1:
  Successfully uninstalled torchaudio-2.5.1
Found existing installation: transformers 4.48.3
Uninstalling transformers-4.48.3:
  Successfully uninstalled transformers-4.48.3
Found existing installation: openai 0.28.0
Uninstalling openai-0.28.0:
  Successfully uninstalled openai-0.28.0
  Using cached torch-2.5.1-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.20.1-cp311-cp311-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.5.1-cp311-cp311-manylinux1_x86_64.whl.metadata (6.4 kB)
  Using cached transformers-4.48.3-py3-none-any.whl.metadata (44 kB)
  Using cached openai-0.28.0-py3-none-any.whl.metadata (13 kB)
Using cached torch-2.5.1

In [2]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import openai
import requests
import cv2
import numpy as np
from PIL import Image
import base64


In [3]:
# ✅ Load BLIP-2 Model for Image Captioning
blip_model_name = "Salesforce/blip2-opt-2.7b"

blip_processor = Blip2Processor.from_pretrained(blip_model_name, force_download=True)
blip_model = Blip2ForConditionalGeneration.from_pretrained(blip_model_name, force_download=True).to("cuda")

print("✅ BLIP-2 Model Loaded Successfully!")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

✅ BLIP-2 Model Loaded Successfully!


In [25]:
from google.colab import files
from PIL import Image

# Upload an image file
uploaded = files.upload()

# Get the uploaded file name
image_path = list(uploaded.keys())[0]

# Open and display the uploaded image
image = Image.open(image_path)
image.show()

print(f"✅ Uploaded Image: {image_path}")


Saving 02d0b720-5780-435a-a623-b3ec13797901.webp to 02d0b720-5780-435a-a623-b3ec13797901.webp
✅ Uploaded Image: 02d0b720-5780-435a-a623-b3ec13797901.webp


In [26]:
def resize_image(image_path, max_size=512):
    """
    Resizes image to a max dimension (512px) to reduce API request size.
    """
    img = Image.open(image_path)
    img.thumbnail((max_size, max_size))
    img.save("resized_image.jpg")  # Save the resized image
    return "resized_image.jpg"

# Resize before sending to BLIP-2
image_path = resize_image(image_path)


In [27]:
def generate_blip_description(image_path):
    """
    Uses BLIP-2 to describe clothing in an image.
    """
    image = Image.open(image_path).convert("RGB")
    inputs = blip_processor(images=image, return_tensors="pt").to("cuda")

    output = blip_model.generate(**inputs)
    description = blip_processor.decode(output[0], skip_special_tokens=True)

    return description

# Get clothing description
description = generate_blip_description(image_path)

print(f"📝 **Generated Clothing Description:** {description}")


📝 **Generated Clothing Description:** a woman wearing an african print dress and yellow purse



In [ ]:
import openai
openai.api_key = ""

def extract_clothing_labels(description):
    """
    Uses GPT-4 to refine BLIP's description into structured clothing labels.
    """
    prompt = f"""
    Given the following detailed clothing description:

    '{description}'

    Extract and list only the specific clothing items (e.g., "Saree", "Salwar Kameez", "Jeans", "T-shirt").
    Do not include unrelated details.
    """

    response = openai.ChatCompletion.create(
        model="gpt-4-turbo",
        messages=[{"role": "system", "content": prompt}]
    )

    return response["choices"][0]["message"]["content"]

# Extract clothing labels
clothing_labels = extract_clothing_labels(description)
print(f"👕 **Extracted Clothing Labels:** {clothing_labels}")


RateLimitError: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.